In [118]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [119]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [120]:
documents = ["https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/01-intro.md", 
             "https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/02-environment.md", 
             "https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/03-rag.md"
            ]

In [121]:
documents_llm = []

for doc in documents:
        documents_llm.append(doc)

len(documents_llm)

3

In [122]:
documents = documents_llm

In [123]:
doc = documents[0]
print(doc)
print(documents)

https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/01-intro.md
['https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/01-intro.md', 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/02-environment.md', 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/03-rag.md']


In [124]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [125]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [126]:
import json

user_prompt = json.dumps(documents)

In [127]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [128]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [129]:
result = response.output_parsed

print(result)

questions=['What is the main idea behind agentic RAG, and how is it different from a basic retrieval-augmented generation setup?', 'Why would you want to use a RAG system instead of just relying on the model’s built-in knowledge?', 'What tools or setup do you need before you can start working through the agentic RAG lessons?', 'How does the retrieval step actually help a language model answer questions more accurately?', 'What are the core pieces of a standard RAG pipeline, from getting documents in to producing the final answer?']


In [130]:
from evaluation_utils import llm_structured

In [131]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What is the main goal of this agentic RAG part of the course, and how is it different from a plain chat-with-docs setup?', 'Why do we need a separate development environment for the project, and what tools or setup does the lesson suggest using?', 'How does a basic RAG pipeline work end to end, from a user question to the final answer?', 'What role do embeddings play in retrieval, and why are they useful for finding relevant chunks?', 'What are the main pieces you need to build a RAG system that can answer questions from your own documents?']


In [132]:
usage.input_tokens, usage.output_tokens

(261, 130)

In [133]:
# question 1 140

In [134]:
import pandas as pd
ground_truth = pd.read_csv("ground-truth.csv")
df_ground_truth = pd.DataFrame(ground_truth)

In [135]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [136]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [137]:
len(chunks)

295

In [138]:
!uv pip install onnxruntime

Using Python 3.12.7 environment at: /Users/donnabrown/Desktop/llm-zoomcamp-2026-code/.venv
Checked 1 package in 9ms


In [139]:
from minsearch import Index
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(chunks)

def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)

from minsearch import VectorSearch
from minsearch import Index
from embedder import Embedder




from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])


def vector_search(query, num_results=5):
    embed = Embedder()
    query_vector = embed.encode(query)
    return vindex.search(query_vector, num_results=num_results)

In [140]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [141]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [142]:
df = pd.read_csv("ground-truth.csv")
ground_truth = df.to_dict(orient="records")
q = ground_truth[0]["question"]

In [143]:
text_search_results = text_search(q, num_results=5)

In [144]:
text_search_results[0]

{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

In [145]:
# question 2 01-agentic-rag/lessons/03-rag.md

In [146]:
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [147]:
from embedder import Embedder

embed = Embedder()

In [148]:
texts = [chunk["content"] for chunk in chunks]

from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i : i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)
X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [149]:
vindex.fit(X, chunks)

In [150]:
vector_search_results = vector_search(q, num_results=5)

In [153]:
vector_search_results[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [152]:
# question 3 01-agentic-rag/lessons/01-intro.md